# Import

In [3]:
import os
import shutil
import re
import pickle
from pathlib import Path

# ---------------- project root setup ----------------
# Pick scratch if it exists, otherwise project
if Path("/vast/palmer/scratch/mcdougal/imc33/mod-extract").exists():
    PROJECT_ROOT = Path("/vast/palmer/scratch/mcdougal/imc33/mod-extract")
else:
    PROJECT_ROOT = Path("/gpfs/gibbs/project/mcdougal/imc33/mod-extract")

log_folder = PROJECT_ROOT / "logs"
run_log_path = log_folder / "run_log.txt"
success_dir = log_folder / "successful"
fail_dir = log_folder / "failed"


# ---------------- utils ----------------
def ensure_dirs(*dirs):
    for d in dirs:
        os.makedirs(d, exist_ok=True)

def parse_run_log(log_path):
    """
    Reads a consolidated run log with lines like:
      ==== Processing <hash>.mod ====
      Successfully processed ...
      ... or ...
      failed: <reason>
    Returns (success_hashes, fail_hashes) based on the NEXT line after the marker.
    """
    success, fail = [], []
    with open(log_path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()
    for i, line in enumerate(lines):
        match = re.match(r"==== Processing (.+)\.mod ====", line.strip())
        if match:
            hash_id = match.group(1)
            next_line = lines[i + 1].strip() if i + 1 < len(lines) else ""
            if "Successfully processed" in next_line:
                success.append(hash_id)
            elif "failed" in next_line.lower():
                fail.append(hash_id)
    return success, fail

def move_logs(hashes, src_dir, dst_dir):
    ensure_dirs(dst_dir)
    for h in hashes:
        src = os.path.join(src_dir, f"{h}.log")
        dst = os.path.join(dst_dir, f"{h}.log")
        if os.path.exists(src):
            shutil.copy2(src, dst)

def copy_failed_mods(failed_log_dir, mod_src_dir, mod_dst_dir):
    failed_hashes = [f.replace(".log", "") for f in os.listdir(failed_log_dir) if f.endswith(".log")]
    copied = 0
    for h in failed_hashes:
        src = os.path.join(mod_src_dir, f"{h}.mod")
        dst = os.path.join(mod_dst_dir, f"{h}.mod")
        if os.path.exists(src):
            shutil.copy2(src, dst)
            copied += 1
        else:
            print(f"⚠️ Missing .mod for {h}")
    return copied

def move_failed_files(nmodl_src, log_src, dst_root):
    nmodl_dst = os.path.join(dst_root, "nmodl")
    log_dst = os.path.join(dst_root, "log")
    ensure_dirs(nmodl_dst, log_dst)

    for f in os.listdir(nmodl_src):
        shutil.move(os.path.join(nmodl_src, f), os.path.join(nmodl_dst, f))
    for f in os.listdir(log_src):
        shutil.move(os.path.join(log_src, f), os.path.join(log_dst, f))

    os.rmdir(nmodl_src)
    os.rmdir(log_src)

    print(f"✅ Moved nmodl-failed → {nmodl_dst}")
    print(f"✅ Moved logs/failed → {log_dst}")

def load_excluded_hashes(pipeline_dir):
    files = ["failed_excluded.pkl", "failed_neither.pkl"]
    excluded = set()
    for fname in files:
        fp = os.path.join(pipeline_dir, fname)
        if os.path.exists(fp):
            with open(fp, "rb") as f:
                excluded.update(pickle.load(f))
    return excluded

# ---------------- new: skip-by-reason filter ----------------
SKIP_PATTERNS = [
    r"suffix\s+not\s+found",   # ValueError: SUFFIX not found ...
    r"exclude_hashes"          # any log mentioning an intentional exclusion
]

def should_skip_failure_for_reason(log_text, patterns=SKIP_PATTERNS):
    text = log_text.lower()
    return any(re.search(pat, text) for pat in patterns)

def read_log_text(log_dir, hash_id):
    fp = os.path.join(log_dir, f"{hash_id}.log")
    if not os.path.exists(fp):
        return ""
    with open(fp, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

def filter_failures_by_log_reason(fail_hashes, log_dir, extra_skip_patterns=None):
    """Remove from fail list any hashes whose logs include skip patterns (e.g., SUFFIX not found)."""
    patterns = SKIP_PATTERNS[:]
    if extra_skip_patterns:
        patterns.extend(extra_skip_patterns)
    keep, skipped = [], []
    for h in fail_hashes:
        txt = read_log_text(log_dir, h)
        if should_skip_failure_for_reason(txt, patterns):
            skipped.append(h)
        else:
            keep.append(h)
    return keep, skipped

# === Main execution ===
#log_folder = "/gpfs/gibbs/project/mcdougal/imc33/mod-extract/logs"
#log_folder = "/vast/palmer/scratch/mcdougal/imc33/mod-extract/logs"
run_log_path = os.path.join(log_folder, "run_log.txt")
success_dir = os.path.join(log_folder, "successful")
fail_dir = os.path.join(log_folder, "failed")
ensure_dirs(success_dir, fail_dir)

# Load and parse hashes
success_hashes, fail_hashes = parse_run_log(run_log_path)

# Remove explicitly excluded hashes first (your existing rule)
pipeline_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "data", "pipeline"))
excluded_hashes = load_excluded_hashes(pipeline_dir)
fail_hashes = [h for h in fail_hashes if h not in excluded_hashes]

# NEW: remove failures that are due to "SUFFIX not found" (or "exclude_hashes") per per-hash logs
fail_hashes_filtered, skipped_reason_hashes = filter_failures_by_log_reason(
    fail_hashes,
    log_folder,
    extra_skip_patterns=None  # add more if needed
)

# Move logs
move_logs(success_hashes, log_folder, success_dir)
move_logs(fail_hashes_filtered, log_folder, fail_dir)

print(f"✅ Moved {len(success_hashes)} successful logs")
print(f"❌ Moved {len(fail_hashes_filtered)} filtered failed logs")
print(f"⏭️ Skipped moving {len(skipped_reason_hashes)} failures (e.g., SUFFIX not found or exclude_hashes)")
if skipped_reason_hashes:
    print(f"   Skipped examples: {skipped_reason_hashes[:5]} ...")

# Copy .mod files from filtered failed logs
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
failed_log_dir = os.path.join(project_root, "logs", "failed")
mod_src_dir = os.path.join(project_root, "data", "raw", "nmodl")
mod_dst_dir = os.path.join(project_root, "data", "raw", "nmodl-failed")
ensure_dirs(mod_dst_dir)

copied_count = copy_failed_mods(failed_log_dir, mod_src_dir, mod_dst_dir)
print(f"✅ Copied {copied_count} .mod files to '{mod_dst_dir}'")


✅ Moved 426 successful logs
❌ Moved 48 filtered failed logs
⏭️ Skipped moving 68 failures (e.g., SUFFIX not found or exclude_hashes)
   Skipped examples: ['009c5ef10778371f25f95c9ad812c6f628c8f2a7bd7f0638d151bce8c9b4ec06', '0c45dc2f28432ff8bfc6cabac9819029179c0bee85e74e3585d0635bb945d3ae', '0dccbeb565ff1bcc5db7b432326368919c37b65933a4fea149d9abb20a77a2cb', '10bceb4b8eddc67187dc443dffd5458371258135295d28015925c3b1e77ac1b6', '11de765f9c0362d05675c37bdf1f96987b9593eb91cda851f0108f26389d5151'] ...
✅ Copied 128 .mod files to '/vast/palmer/scratch/mcdougal/imc33/mod-extract/data/raw/nmodl-failed'


In [8]:
SCRIPT_DIR = Path.cwd()

import os
import shutil
import re
import pickle
from pathlib import Path


# ---------------- project & output roots ----------------
# Pick scratch if it exists, otherwise project
if Path("/vast/palmer/scratch/mcdougal/imc33/mod-extract").exists():
    PROJECT_ROOT = Path("/vast/palmer/scratch/mcdougal/imc33/mod-extract")
else:
    PROJECT_ROOT = Path("/gpfs/gibbs/project/mcdougal/imc33/mod-extract")

# Where the consolidated logs live
log_folder = PROJECT_ROOT / "logs"
run_log_path = log_folder / "run_log.txt"
success_dir = log_folder / "successful"

# Where the .mod sources live
mod_src_dir = PROJECT_ROOT / "data" / "raw" / "nmodl"

# Bundle failed outputs under the *script directory*/failed/<hash>/
# (This yields: code/failed/<hash>/<hash>.log and <hash>.mod)
SCRIPT_DIR = Path.cwd()
FAILED_BUNDLE_ROOT = SCRIPT_DIR / "failed"

# If space is tight, symlink instead of copying the .log/.mod into each bundle.
USE_SYMLINKS = False

# ---------------- utils ----------------
def ensure_dirs(*dirs):
    for d in dirs:
        Path(d).mkdir(parents=True, exist_ok=True)

def parse_run_log(log_path):
    success, fail = [], []
    with open(log_path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()
    for i, line in enumerate(lines):
        m = re.match(r"==== Processing (.+)\.mod ====", line.strip())
        if m:
            h = m.group(1)
            nxt = lines[i + 1].strip() if i + 1 < len(lines) else ""
            if "Successfully processed" in nxt:
                success.append(h)
            elif "failed" in nxt.lower():
                fail.append(h)
    return success, fail

def move_logs(hashes, src_dir, dst_dir):
    """For successful logs only; we still keep your success layout."""
    ensure_dirs(dst_dir)
    for h in hashes:
        src = Path(src_dir) / f"{h}.log"
        dst = Path(dst_dir) / f"{h}.log"
        if src.exists():
            shutil.copy2(src, dst)

def load_excluded_hashes(pipeline_dir):
    files = ["failed_excluded.pkl", "failed_neither.pkl"]
    excluded = set()
    for fname in files:
        fp = Path(pipeline_dir) / fname
        if fp.exists():
            with open(fp, "rb") as f:
                excluded.update(pickle.load(f))
    return excluded

# ---- reason-based skipping (e.g., SUFFIX not found) ----
SKIP_PATTERNS = [
    r"suffix\s+not\s+found",
    r"exclude_hashes",
]

def should_skip_failure_for_reason(log_text, patterns=SKIP_PATTERNS):
    t = log_text.lower()
    return any(re.search(p, t) for p in patterns)

def read_log_text(log_dir, hash_id):
    fp = Path(log_dir) / f"{hash_id}.log"
    if not fp.exists():
        return ""
    with open(fp, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

def filter_failures_by_log_reason(fail_hashes, log_dir, extra_skip_patterns=None):
    pats = SKIP_PATTERNS[:] + (extra_skip_patterns or [])
    keep, skipped = [], []
    for h in fail_hashes:
        if should_skip_failure_for_reason(read_log_text(log_dir, h), pats):
            skipped.append(h)
        else:
            keep.append(h)
    return keep, skipped

# ---- NEW: bundle failed items under code/failed/<hash>/ ----
def _safe_link_or_copy(src: Path, dst: Path):
    if not src.exists():
        return False
    if dst.exists():
        return True  # already present
    if USE_SYMLINKS:
        try:
            dst.symlink_to(src)
        except FileExistsError:
            pass
    else:
        shutil.copy2(src, dst)
    return True

def bundle_failed_hash(hash_id, log_src_dir: Path, mod_src_dir: Path, bundle_root: Path):
    """
    Create: bundle_root/<hash>/
      - <hash>.log (from log_src_dir)
      - <hash>.mod (from mod_src_dir)
    Returns (has_log, has_mod)
    """
    bundle_dir = bundle_root / hash_id
    ensure_dirs(bundle_dir)

    log_src = Path(log_src_dir) / f"{hash_id}.log"
    mod_src = Path(mod_src_dir) / f"{hash_id}.mod"

    log_dst = bundle_dir / f"{hash_id}.log"
    mod_dst = bundle_dir / f"{hash_id}.mod"

    has_log = _safe_link_or_copy(log_src, log_dst)
    has_mod = _safe_link_or_copy(mod_src, mod_dst)

    missing = []
    if not has_log: missing.append("log")
    if not has_mod: missing.append("mod")
    if missing:
        print(f"⚠️  {hash_id}: missing {', '.join(missing)}")

    return has_log, has_mod

def bundle_all_failed(fail_hashes, log_src_dir: Path, mod_src_dir: Path, bundle_root: Path):
    ensure_dirs(bundle_root)
    bundled = 0
    for h in fail_hashes:
        has_log, has_mod = bundle_failed_hash(h, log_src_dir, mod_src_dir, bundle_root)
        if has_log or has_mod:
            bundled += 1
    return bundled

# ================ Main ================
ensure_dirs(success_dir, FAILED_BUNDLE_ROOT)

success_hashes, fail_hashes = parse_run_log(run_log_path)

# Remove explicitly excluded
pipeline_dir = PROJECT_ROOT / "data" / "pipeline"
excluded_hashes = load_excluded_hashes(pipeline_dir)
fail_hashes = [h for h in fail_hashes if h not in excluded_hashes]

# Remove failures due to SUFFIX-not-found / exclude_hashes
fail_hashes_filtered, skipped_reason_hashes = filter_failures_by_log_reason(
    fail_hashes, log_folder, extra_skip_patterns=None
)

# 1) Handle successes as before
move_logs(success_hashes, log_folder, success_dir)

# 2) NEW: Bundle fails: code/failed/<hash>/{<hash>.log,<hash>.mod}
bundled_count = bundle_all_failed(
    fail_hashes_filtered,
    log_src_dir=log_folder,
    mod_src_dir=mod_src_dir,
    bundle_root=FAILED_BUNDLE_ROOT,
)

print(f"✅ Moved {len(success_hashes)} successful logs → {success_dir}")
print(f"📦 Bundled {bundled_count} failed hashes → {FAILED_BUNDLE_ROOT}")
print(f"⏭️ Skipped {len(skipped_reason_hashes)} failures (SUFFIX not found / exclude_hashes)")

✅ Moved 426 successful logs → /vast/palmer/scratch/mcdougal/imc33/mod-extract/logs/successful
📦 Bundled 48 failed hashes → /vast/palmer/scratch/mcdougal/imc33/mod-extract/code/failed
⏭️ Skipped 68 failures (SUFFIX not found / exclude_hashes)


In [4]:
!git add .
!git commit -m "use relative path"
!git push

[main 9bed37f] use relative path
 1 file changed, 78 insertions(+), 26 deletions(-)
Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 64 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 1.92 KiB | 1.92 MiB/s, done.
Total 4 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To github.com:innacohen/mod-extract.git
   fe4457f..9bed37f  main -> main
